# 34. Reconcile a target table against its source

**Track:** pyspark | **Difficulty:** hard | **Tags:** reconciliation, data validation, full outer join, null-safe comparison, eqNullSafe

Run the three setup cells below once, in order. Then read the problem, write your
answer in the answer cell so that it ends with a DataFrame called `result`, and run
`check(result)`. You can run the check as many times as you like.

Free Colab runs only two or three notebooks at a time. When you are done here, end the
session with Runtime > Disconnect and delete runtime before opening the next problem.

In [ ]:
# Setup: installs PySpark on Colab and starts a local Spark session (about a minute).
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.1.1"

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )
    # PySpark 4 needs Java 17 or newer.
    java_major = 0
    if shutil.which("java"):
        version_text = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        match = re.search(r'version "(\d+)', version_text)
        java_major = int(match.group(1)) if match else 0
    if java_major < 17:
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder.master("local[2]")
    .appName("034-reconcile-source-and-target")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready.")

In [ ]:
#@title Load the data (run this cell, no need to read it)
import base64
import pathlib

_DATA = {
    "source_customers": """
UEFSMRUEFeASFdIJTBWsAhUAEgAAsAkIRAQABQEAIg0IAEYNCABSDQgACA0IBPEDBScEAG4NEAAEDQgATw0IAFcNCAB+DQgAWQ0I
AAoNCAADDQgAZw0IAHoNCABlDQgAGg0IAFANCABcDQgA/Q14AEkNEAByDQgA8w0YAG8NEAD8DRAAQw0QAPgNEABODRAAHw0IAOwN
GAA0DRAARw0IAA8NCAATDQgAdA0IAGYNCAB2DQgALw0IAOkNSAAVDRAAPA0IAAANCABYDQgAMA0IAC0NCAAUDQgA9w1AAGMNEADy
DRAAPg0QAE0NCADwDRgAJw0QACsNCAAgDQgAGA0IAHENCAAcDQgAZA0IACUNCAAdDQgAIw0IAEUNCAD+DWAACw0QAGwNCAAsDQgA
Nw0IAC4NCAB8DQgAAg0IAP8NQAAeDRAAEg0IAGsNCAD0DSAA+Q0IABcNGAAzDQgA7Q0YACENEAAQDQgA7g0YADsNEAB3DQgA9Q0Y
ADkNEAD6DRAAbQ0QAHANCAA/DQgACQ0IADgNCAABDQgAOg0IAAUNCABhDQgAUQ0IAAYNCABUDQgAew0IACgNCABbDQgEagRpGADv
DYgASA0QAEwNCABADQgAMQ0IAOoNKAAMDRAAaQ0IAD0NCABTDQgAdQ0IAHkNCABCDQgADg0IAGINCAAWDQgAcw0IABkNCAAHDQgE
9gMJoAB9DRAAVg0IAEsNCAAbDQgANg0IAF0NCAApDQgAVQ0IADUNCABBDQgAWg0IAPsNYAARDRAAMg0IAGANCADrDSAAeA0QAF4N
CAAqDQgAXw0IACQNCAANDQgAaA0IPCYEAAAAAAAASgQAAAAAAAAVABXCAhXKAiwVrAIVEBUGFQYcGAh+BAAAAAAAABgI6QMAAAAA
AAAWACgIfgQAAAAAAAAYCOkDAAAAAAAAEREAAAChAfCgAwAAAKwCAQgnAAECAwQFBgcICQoLDA0ODxAREhMUFRYXGBkaGxwdHh8g
ISIjJCUmJygpKissLS4vMDEyMzQ1Njc4OTo7PD0+P0BBQkNERUZHSElKS0xNTk9QUVJTVFVWV1hZWltcXV5fYGFiY2RlZmdoaWpr
bG1ub3BxcnN0dXZ3eHl6e3x9fn+AgYKDhIWGh4iJiouMjY6PkJGSk5SVAAAVBBXgMxWkC0wVlAIVABIAAPAZXBQAAAB1c2VyMTA5
MkBleGFtcGxlLmNvbR0YADRSGAAIMTA2UhgABDAzWkgABDA5UjAABDEzXmAAADNWMAAEMTFWGAAENTBWGAAAMVZIAAAwWngACDAy
N1ZIAF4YAAA0VggBCDEyNVJIAAAwXqgAADBaCAEAMVZgAAQwMlYIAQQwOVrAAAQzOFaQAF5QAQAzWsAAADJWaAEEMDlWkAAEMTBa
EAIANVpgAFoIAQQwN1oIAQA5WkgAADNWcAIEMDRWEAIEMTRW2AAEMTJWeAAEMTRa2AAAN1YIAQQwMFoYAAA0WsAAADhWEAIEMDJW
GAAEMTFekABaGAAANl4gAVZgAAQwMVaoAAQxMlZQAQQwOFo4AV4IAQAwWqACADZaYAAANlboAgQwNVZ4AAQwNFpgAAA1ViABADFa
aAEEMDZayAEANVqoAAA1WmgBADlaMAAAMlqQAF5YBQA2WtgAADda0AJeCAFa6AIAMF7gAQA1WigCWhgDBDEzWjgBXrgCADFa4AEA
NFoYAAA3VqACBDAwWhgAADVaSABe2AYAOFawAQAxWlAEADBeeAYAOF4IAVbIAQQxM1p4AAAzWugCADha2ABaSAAEMDhWKAIAMF7A
BgA4VqACBDAyVugCADFaqAYAMV6YAQAzVpAAADFaIAQAMV4QAgA2VqACADFeyARaeAAEMDBaUAEAOVaAAQQxMFbAAAQwOFbgAQQw
N1bgAQAwWjgHADFaaAEEMDhW6AIAMVrAAAQxNFqIAgA5WsAAWkAIADFasAQEMDRaIAFimAdWQAIAMGI4BFrgAVpIAAQxMVrYAAA5
WngAADVaIAEAN1rIAQA2XoABWmAAADdaWAIAOFYwAAAxWvAABDAxVjAAADBe4AEAN1o4AVrYCQAxWhAIADFasAQEMDZa+AFeqAAA
NlqYAQAzViABBDEyXoABWpgENDk4QGV4YW1wbGUuY29tFQAV4AIV6AIsFawCFRAVBhUGHDYYKBR1c2VyMTE1MEBleGFtcGxlLmNv
bRgUdXNlcjEwMDFAZXhhbXBsZS5jb20REQAAALAB8K8aAAAAA30mAQP+HAEH/v72EgED/ioBA94WAQf++w8IJQABAgMEBQYHCAkK
CwwNDg8QERITFBUWFxgZGhscHR4fICEiIyQlJicoKSorLC0uLzAxMjM0NTY3ODk6Ozw9Pj9AQUJDREVGR0hJSktMTU5PUFFSU1RV
VldYWVpbXF1eX2BhYmNkZWZnaGlqa2xtbm9wcXJzdHV2d3h5ent8fX5/gIGCg4SFhoeIiQAAAAAAABUEFaoBFagBTBUQFQASAABV
UAYAAABKYWlwdXIHAAAAQ2hlbm5haQELbEtvbGthdGEEAAAAUHVuZQkAAABIeWRlcmFiYWQBNRRNdW1iYWkBF0RCZW5nYWx1cnUF
AAAARGVsaGkVABWgARWmASwVrAIVEBUGFQYcNiQoBFB1bmUYCUJlbmdhbHVydRERAAAAUPBPFwAAACIBEf73H69//t+dEgED/hIB
B3777yYBAyOIFiaq6qWE5MIDm2CrTThOOIUkkDKObacMlDZ2Upd/oUe7g6rG1wac7Nm+CPrS1MlQDgAVBBUyFTZMFQoVABIAABlg
AABMS0AAAAehIAAAmJaAAAAmJaAAAA9CQBUAFZoBFaABLBWsAhUQFQYVBhwYBQAAmJaAGAUAAAehIBYUKAUAAJiWgBgFAAAHoSAR
EQAAAE3wTBEAAAAgAQX8WxQBCf77f3+UAQEDAgMliLSIChmIIKaNQBhiIAFxGUkBFIlMCJmMk8KMYgIMXJIgAChGGJQgo5hpiaIo
WCCKVCQxBAkAFQQZXDUAGAZzY2hlbWEVCAAVBCUCGAtjdXN0b21lcl9pZAAVDCUCGAVlbWFpbCUATBwAAAAVDCUCGARjaXR5JQBM
HAAAABUOFQoVAhgMY3JlZGl0X2xpbWl0JQoVBBUULFwVBBUUAAAAFqwCGRwZTCYAHBUEGTUABhAZGAtjdXN0b21lcl9pZBUCFqwC
FsgWFsINJvwJJggcGAh+BAAAAAAAABgI6QMAAAAAAAAWACgIfgQAAAAAAAAYCOkDAAAAAAAAEREAGSwVBBUAFQIAFQAVEBUCADwp
BhkmAKwCAAAAJgAcFQwZNQAGEBkYBWVtYWlsFQIWrAIW7jcWug8mkBkmyg0cNhgoFHVzZXIxMTUwQGV4YW1wbGUuY29tGBR1c2Vy
MTAwMUBleGFtcGxlLmNvbRERABksFQQVABUCABUAFRAVAgA8FpArGQYZJhiUAgAAACYAHBUMGTUABhAZGARjaXR5FQIWrAIWwAMW
xAMmzB4mhB0cNiQoBFB1bmUYCUJlbmdhbHVydRERABksFQQVABUCABUAFRAVAgA8FqYOGQYZJiSIAgAAACYAHBUOGTUABhAZGAxj
cmVkaXRfbGltaXQVAhasAhbUAhbeAiaaISbIIBwYBQAAmJaAGAUAAAehIBYUKAUAAJiWgBgFAAAHoSAREQAZLBUEFQAVAgAVABUQ
FQIAPCkGGSYUmAIAAAAWylQWrAImCBaeIwAZHBgMQVJST1c6c2NoZW1hGIwDLy8vLy95QUJBQUFRQUFBQUFBQUtBQXdBQmdBRkFB
Z0FDZ0FBQUFBQkJBQU1BQUFBQ0FBSUFBQUFCQUFJQUFBQUJBQUFBQVFBQUFDMEFBQUFkQUFBQUVnQUFBQUVBQUFBYlAvLy93QUFB
UWNRQUFBQUtBQUFBQVFBQUFBQUFBQUFEQUFBQUdOeVpXUnBkRjlzYVcxcGRBQUFBQUFJQUF3QUJBQUlBQWdBQUFBS0FBQUFBZ0FB
QUt6Ly8vOEFBQUVGRUFBQUFCZ0FBQUFFQUFBQUFBQUFBQVFBQUFCamFYUjVBQUFBQU5qLy8vL1UvLy8vQUFBQkJSQUFBQUFjQUFB
QUJBQUFBQUFBQUFBRkFBQUFaVzFoYVd3QUFBQUVBQVFBQkFBQUFCQUFGQUFJQUFZQUJ3QU1BQUFBRUFBUUFBQUFBQUFCQWhBQUFB
QWtBQUFBQkFBQUFBQUFBQUFMQUFBQVkzVnpkRzl0WlhKZmFXUUFDQUFNQUFnQUJ3QUlBQUFBQUFBQUFVQUFBQUE9ABggcGFycXVl
dC1jcHAtYXJyb3cgdmVyc2lvbiAyNS4wLjEZTBwAABwAABwAABwAAADvAwAAUEFSMQ==
""",
    "target_customers": """
UEFSMRUEFcASFcQJTBWoAhUAEgAAoAkIMwQABQEAcA0IAEINCAT1AwUXBABtDRAAOQ0IAFANCAD/DSAAdQ0QAFgNCAAYDQgA/Q0g
AA0NEAAnDQgAHA0IACgNCAD2DSgAPA0QAPQNEAAlDRAAeQ0IADcNCAB8DQgACw0IBGoECagAPw0IABUNCAA+DQgAWQ0IAFQNCADx
DWAAIQ0QAEoNCABJDQgAdw0IAEwNCABoDQgAGg0IAAINCAAbDQgAVg0IAA4NCAARDQgE1AcJmAD7DXAAbw0YAB0NCADzDRgAaw0Q
AP4NEAA4DRAA7g0QAC4NEABBDQgAYQ0IAH4NCAAUDQgAdA0IAGkNCABADQgAKQ0IAHENCAAkDQgADw0IACwNCATvAwmwAAUNEAAq
DQgANA0IAPcNIAArDRAAFw0IACMNCAB2DQgAXQ0IAEcNCAAtDQgAHg0IAEQNCABkDQgACg0IAHINCABlDQgAEg0IAFcNCAAHDQgA
Fg0IAOoNkABaDRAA7Q0QAD0NEAD4DRAAAA0QAAkNCAAvDQgAYA0IAFENCAAMDQgA6Q04AGYNEABfDQgA/A0YAE0NEABiDQgABg0I
AH0NCAAEDQgANQ0IAG4NCABLDQgA0k0YAHsNEAAyDQgAYw0IAPoNaABsDRAAEw0IAE4NCADwDSAA6w0IADYNGAA6DQgAOw0IAB8N
CAABDQgAXg0IAAgNCAAZDQgAMA0IAE8NCAD5DVgARQ0QAHoNCADRDbgARg0QACYNCAB4DQgAXA0IACANCABIDQgA8g1QADENEABS
DQgAZw0IAAMNCABVDQg8EAQAAAAAAADTBwAAAAAAABUAFcICFcoCLBWwAhUQFQYVBhwYCNQHAAAAAAAAGAjpAwAAAAAAABYAKAjU
BwAAAAAAABgI6QMAAAAAAAAREQAAAKEB8KADAAAAsAIBCCcAAQIDBAUGBwgJCgsMDQ4PBRAREhMUFRYXGBkaGxwdHh8gISIjJCUm
JygpKissLS4vMDEyMzQ1Njc4KDk6Ozw9Pj9AQUJDREVGR0hJSktMTU5PUFFSU1RVVldYWVpbXF1eX2BhYmNkZWZnaGlqa0psbW5v
anBxcnN0dXZ3eHl6e3x9fn+AgYKDhIWGh4iJiouMjY6PkJGSkxUEFdAyFZIMTBWOAhUAEgAAqBlcFAAAAHVzZXIxMDc1QGV4YW1w
bGUuY29tFRgIMTM2UhgACDA5MFYYAAQxM1IYAAQxM1YYAAgwODFWMAAEMDRSGAAEMDJWSAAEMTRaSAAEMTJWSAAENDhWGAAAMlZI
AAgwMzdWMAAANlaQAAQwNVp4AAA2WtgAADFaGAAAOFoYAF7YAAA2VsAAADFa8AAEMTNayAEAOFrwAAA0VigCBDA4VigCADFeEAIA
MFqAAQQwOVZoAQA1WpAAADlaSAAAOVYwAAQxNFawAQQxMFYIAQQxMkJgAExVU0VSMTA1MEBFWEFNUExFLkNPTXUwBDAyOQgIb3Jn
GRgENTEV2C4YAAQxMVZ4AAQwM0J4AGmoVugCBDIwXjADADFWUAEEMTNayAEANVYgAQQwMRnANiAEVhgABDAyeQAu8AAEMDha8AAA
MDk4LmAABDA3WjAAADha2ABawAMEMTVaSAAANHlgLqgABDE0GTAyGAAAMlZ4AAQwOFqwAQA2WmgBADYZYDLwAAAzWmAAADZaYAAA
MFboAgAwXqgAADZeaAFaGAAAMV7AAFp4AAA0WhgAADVW2AAEMTRZKDIIAQA5XngAVkgABDA1Ocg2SABWYAAEMTJCMABl2AgxMzhS
2AMEMTJakABawAAAMVowAwAwfTAuWAIEMDRagAEAMFrAAF4wBgAwWqgAADhaGABGCAGJaF6wBwA3VvADBDEyVtACADFekAAAMF5I
AFYIAQAxWsgEADBieABeYABWUAEAMFoIBwQxNFpwAgA3VugCBDEzQlgCKVAAOVJIAAAyYuABXkgDADdWYAAAMVoYCQQwMVY4BAQx
M1oIAVogBwAwXrAHADdWYAAEMDhaYAAAOFpgBgA1XqACVhgAADFa2AAAMGLYAFKAAQgxMDdWqAAEMTBaqAAAMVb4AQQwOVYwAAAx
VtgDADJeAAMEMDlW4AEEMDZaqADdwC74BAQxMVZIAwQwNVoYAAA5WhgAADda2ABasAcEMTJaIAFaGAAAMVbwCTwyMDAzQGV4YW1w
bGUuY29tFQAV0gIV2gIsFbACFRAVBhUGHDYeKBR1c2VyMjAwNEBleGFtcGxlLmNvbRgUVVNFUjEwNTBARVhBTVBMRS5DT00REQAA
AKkB8KgTAAAAKgED9EQBEf7f97/vX+/rIgEDvgglAAECAwQFBgcICQoLDA0ODwUQERITFBUWFxgZGhscHR4fICEiIyQlJicoKSor
LC0uLzAxMjM0NSU2Nzg5Ojs8PT4/QEFCQ0RFRkdISUpLTE1OT1BRUlNUVVZXWFlaW1xdXl9gYWJjZGVmZ2hpamtsbW5vcHFyc3R1
dnd4eXp7fH1+f4CBgoOEhYYAAAAAAAAAFQQVqgEVrgFMFRAVABIAAFXwPgkAAABCZW5nYWx1cnUGAAAATXVtYmFpBwAAAENoZW5u
YWkGAAAASmFpcHVyBAAAAFB1bmUHAAAAS29sa2F0YQE/REh5ZGVyYWJhZAUAAABEZWxoaRUAFaABFaYBLBWwAhUQFQYVBhw2IigE
UHVuZRgJQmVuZ2FsdXJ1EREAAABQ8E8XAAAAA/geAQPuNAEPvr/e73/XfxABBf73HgEDI4gmIp1MwVoAwJF1QA3A/oBmOdi6Lweh
UA6wbIvTxb2XFiD1zxCrSmKJK4/YOa1aT6XYGBUEFWQVZkwVFBUAEgAAMnQAACYloAAAB6EgAACYloAAAA9CQAAATEtAAAExLQAB
HjzSAAAPQnIAAJiWsgAAHoSAFQAVygEVzgEsFbACFRAVBhUGHBgFAAExLQAYBQAAB6EgFhIoBQABMS0AGAUAAAehIBERAAAAZUQX
AAAAMAED/kwBBf69EAED/hYBCfBOFAEF/h8EJQAhAwQiQkQzEBQxARU0MyEDRCMzYRMDABIRMTQTFBIzcRQDFEIwQBQiIRAyQDFB
NBAjQjICIjIyNENCFCEQg0kTIjMwIUI0AxUEGVw1ABgGc2NoZW1hFQgAFQQlAhgLY3VzdG9tZXJfaWQAFQwlAhgFZW1haWwlAEwc
AAAAFQwlAhgEY2l0eSUATBwAAAAVDhUKFQIYDGNyZWRpdF9saW1pdCUKFQQVFCxcFQQVFAAAABawAhkcGUwmABwVBBk1AAYQGRgL
Y3VzdG9tZXJfaWQVAhawAhaoFha0DSbuCSYIHBgI1AcAAAAAAAAYCOkDAAAAAAAAFgAoCNQHAAAAAAAAGAjpAwAAAAAAABERABks
FQQVABUCABUAFRAVAgA8KQYZJgCwAgAAACYAHBUMGTUABhAZGAVlbWFpbBUCFrACFtA2FpoQJvAZJrwNHDYeKBR1c2VyMjAwNEBl
eGFtcGxlLmNvbRgUVVNFUjEwNTBARVhBTVBMRS5DT00REQAZLBUEFQAVAgAVABUQFQIAPBboKhkGGSYekgIAAAAmABwVDBk1AAYQ
GRgEY2l0eRUCFrACFsADFsoDJqQfJtYdHDYiKARQdW5lGAlCZW5nYWx1cnUREQAZLBUEFQAVAgAVABUQFQIAPBa2DhkGGSYijgIA
AAAmABwVDhk1AAYQGRgMY3JlZGl0X2xpbWl0FQIWsAIWtgMWvAMmoiImoCEcGAUAATEtABgFAAAHoSAWEigFAAExLQAYBQAAB6Eg
EREAGSwVBBUAFQIAFQAVEBUCADwpBhkmEp4CAAAAFu5TFrACJggW1CQAGRwYDEFSUk9XOnNjaGVtYRiMAy8vLy8veUFCQUFBUUFB
QUFBQUFLQUF3QUJnQUZBQWdBQ2dBQUFBQUJCQUFNQUFBQUNBQUlBQUFBQkFBSUFBQUFCQUFBQUFRQUFBQzBBQUFBZEFBQUFFZ0FB
QUFFQUFBQWJQLy8vd0FBQVFjUUFBQUFLQUFBQUFRQUFBQUFBQUFBREFBQUFHTnlaV1JwZEY5c2FXMXBkQUFBQUFBSUFBd0FCQUFJ
QUFnQUFBQUtBQUFBQWdBQUFLei8vLzhBQUFFRkVBQUFBQmdBQUFBRUFBQUFBQUFBQUFRQUFBQmphWFI1QUFBQUFOai8vLy9VLy8v
L0FBQUJCUkFBQUFBY0FBQUFCQUFBQUFBQUFBQUZBQUFBWlcxaGFXd0FBQUFFQUFRQUJBQUFBQkFBRkFBSUFBWUFCd0FNQUFBQUVB
QVFBQUFBQUFBQkFoQUFBQUFrQUFBQUJBQUFBQUFBQUFBTEFBQUFZM1Z6ZEc5dFpYSmZhV1FBQ0FBTUFBZ0FCd0FJQUFBQUFBQUFB
VUFBQUFBPQAYIHBhcnF1ZXQtY3BwLWFycm93IHZlcnNpb24gMjUuMC4xGUwcAAAcAAAcAAAcAAAA7wMAAFBBUjE=
""",
}

_data_dir = pathlib.Path("data") / "034-reconcile-source-and-target"
_data_dir.mkdir(parents=True, exist_ok=True)
for _name, _encoded in _DATA.items():
    (_data_dir / f"{_name}.parquet").write_bytes(base64.b64decode(_encoded))

source_customers = spark.read.parquet((_data_dir / "source_customers.parquet").resolve().as_posix())
source_customers.createOrReplaceTempView("source_customers")
target_customers = spark.read.parquet((_data_dir / "target_customers.parquet").resolve().as_posix())
target_customers.createOrReplaceTempView("target_customers")

for _name in _DATA:
    print(f"{_name}: {spark.table(_name).count()} rows")

In [ ]:
#@title Grader (run this cell, no need to read it)
"""Result grading shared by the build script and every generated notebook.

build_problem.py pastes this file into the notebook verbatim, so it must stay
standalone: standard library only, no imports from the rest of the toolkit.

A result is reduced to a "summary": its columns and types, its row count and
SHA-256 fingerprints of its normalized rows. Only the summary of the correct
answer is shipped in a notebook, never the rows themselves.
"""
import datetime
import decimal
import hashlib
import json
import math

FLOAT_DECIMALS = 6


def normalize_value(value):
    """Turn one cell value into something json.dumps renders the same way everywhere."""
    if value is None or isinstance(value, (bool, int, str)):
        return value
    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
        return round(value, FLOAT_DECIMALS) + 0.0  # + 0.0 turns -0.0 into 0.0
    if isinstance(value, decimal.Decimal):
        return format(value, "f")
    if isinstance(value, (datetime.datetime, datetime.date)):
        return value.isoformat()
    if isinstance(value, (bytes, bytearray)):
        return bytes(value).hex()
    if hasattr(value, "asDict"):  # pyspark Row (struct column)
        return {key: normalize_value(item) for key, item in value.asDict().items()}
    if isinstance(value, dict):  # map column
        pairs = [[normalize_value(key), normalize_value(item)] for key, item in value.items()]
        return sorted(pairs, key=_encode)
    if isinstance(value, (list, tuple)):
        return [normalize_value(item) for item in value]
    return str(value)


def _encode(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


def _digest(lines):
    sha = hashlib.sha256()
    for line in lines:
        sha.update(line.encode("utf-8"))
        sha.update(b"\n")
    return sha.hexdigest()


def describe_schema(df):
    """Column names and Spark types, sorted by name so column order is ignored."""
    return sorted([field.name, field.dataType.simpleString()] for field in df.schema.fields)


def summarize_rows(columns, rows, order_matters=False):
    """columns: sorted [name, type] pairs. rows: normalized values in that column order."""
    encoded = [_encode(row) for row in rows]
    if not order_matters:
        encoded.sort()
    column_fingerprints = {}
    for index, (name, _) in enumerate(columns):
        column_fingerprints[name] = _digest(sorted(_encode(row[index]) for row in rows))
    return {
        "columns": [list(column) for column in columns],
        "row_count": len(rows),
        "order_matters": order_matters,
        "fingerprint": _digest([_encode(columns)] + encoded),
        "column_fingerprints": column_fingerprints,
    }


def summarize(df, order_matters=False):
    columns = describe_schema(df)
    names = [name for name, _ in columns]
    # collect() returns `timestamp` values in the machine's local time zone, which differs
    # between the build machine and Colab. Casting to string inside Spark uses the session
    # time zone instead (UTC in both places). `timestamp_ntz` columns are not affected.
    expressions = [
        "CAST(`%s` AS STRING) AS `%s`" % (name, name) if kind == "timestamp" else "`%s`" % name
        for name, kind in columns
    ]
    collected = df.selectExpr(*expressions).collect()
    rows = [[normalize_value(row[name]) for name in names] for row in collected]
    return summarize_rows(columns, rows, order_matters)


def grade(df, expected):
    """Compare a DataFrame to the expected summary. Returns (passed, hints).

    The hints say what kind of thing is wrong (columns, types, row count, which
    column) but never what the right values are.
    """
    if not hasattr(df, "schema") or not hasattr(df, "collect"):
        return False, ["check() needs a Spark DataFrame, got %s." % type(df).__name__]

    names = [field.name for field in df.schema.fields]
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        return False, ["Your result has duplicate column names: %s." % ", ".join(duplicates)]

    actual = dict(describe_schema(df))
    wanted = dict(expected["columns"])
    hints = []
    missing = sorted(set(wanted) - set(actual))
    extra = sorted(set(actual) - set(wanted))
    if missing:
        hints.append("Missing columns: %s." % ", ".join(missing))
    if extra:
        hints.append("Unexpected columns: %s." % ", ".join(extra))
    for name in sorted(set(wanted) & set(actual)):
        if wanted[name] != actual[name]:
            hints.append("Column %s has type %s, expected %s." % (name, actual[name], wanted[name]))
    if hints:
        return False, hints

    row_count = df.count()
    if row_count != expected["row_count"]:
        return False, [
            "Row count is wrong: your result has %d rows, expected %d."
            % (row_count, expected["row_count"])
        ]

    summary = summarize(df, expected["order_matters"])
    if summary["fingerprint"] == expected["fingerprint"]:
        return True, []

    wrong = sorted(
        name
        for name, fingerprint in summary["column_fingerprints"].items()
        if fingerprint != expected["column_fingerprints"][name]
    )
    if wrong:
        return False, ["Columns and row count are right, but values are wrong in: %s." % ", ".join(wrong)]
    if expected["order_matters"]:
        return False, ["The values are right, but the rows are in the wrong order."]
    return False, ["Each column has the right set of values, but they are combined into the wrong rows."]


def make_check(expected):
    def check(df):
        passed, hints = grade(df, expected)
        if passed:
            print("Correct. Your result matches the expected output.")
        else:
            print("Not correct yet.")
            for hint in hints:
                print("  - " + hint)
        # Returns nothing, so a notebook cell ending in check(result) prints only the message.

    return check

EXPECTED = {'column_fingerprints': {'customer_id': '61b2d06dc5d6581e4e92e483a42176495b4bbf999255e0d56a3e8151aa7fd136',
                         'issue': '272a27b3681dbb8cdd0800a29f1061608fd459b0179525117124d1c418a7d43d',
                         'mismatched_columns': 'd0cc9c2150f890b57443a6e6a12b745f16ab5eda0367722b141a25c1cf5534ed'},
 'columns': [['customer_id', 'bigint'], ['issue', 'string'], ['mismatched_columns', 'string']],
 'fingerprint': '91db219f7b5517c0a74f8f066d209ddb7c9fb4a59b57ca61948be40c5b67c392',
 'order_matters': False,
 'row_count': 35}

check = make_check(EXPECTED)

## The situation

A nightly job copies customers from the source system into the warehouse. The only
check on it compares row counts, and the counts are close enough that nobody looks
further.

Finance has found a customer whose credit limit in the warehouse is not the one in
the source. You are asked for a proper reconciliation: every customer where the two
tables disagree, and how.

## Input tables

### `source_customers`

One row per customer. `customer_id` is unique and never NULL.

### `target_customers`

The loaded copy. `customer_id` is never NULL, but it is not guaranteed to be unique.

Both tables have the same columns:

| Column | Type | Description |
|---|---|---|
| `customer_id` | BIGINT | The key. |
| `email` | STRING | Can be NULL. |
| `city` | STRING | Can be NULL. |
| `credit_limit` | DECIMAL(10,2) | Can be NULL. |

## Requirement

Return one row for every `customer_id` that has a problem. A customer with no
problem does not appear. Check in this order and report the first that applies:

1. `MISSING_IN_TARGET`: the id is in the source and not in the target.
2. `MISSING_IN_SOURCE`: the id is in the target and not in the source.
3. `DUPLICATE_IN_TARGET`: the id is in the source, and has more than one row in the
   target. The column values are not compared for these.
4. `MISMATCH`: the id has one row on each side and at least one of `email`, `city`,
   `credit_limit` differs.

Comparing values:

- Values are compared exactly. `Pune` and `pune` are different.
- NULL on both sides is a match. NULL on one side and a value on the other is a
  difference.

`mismatched_columns` lists the columns that differ, separated by commas with no
spaces, always in the order `email`, `city`, `credit_limit`. For example
`email,credit_limit`. It is NULL for every issue other than `MISMATCH`.

## Expected output

One row per customer with a problem.

| Column | Type | Description |
|---|---|---|
| `customer_id` | BIGINT | |
| `issue` | STRING | One of the four values above. |
| `mismatched_columns` | STRING | Only for `MISMATCH`, otherwise NULL. |

In [ ]:
source_customers.show(5)
target_customers.show(5)

## Your answer

Use the DataFrame API. The tables are available as DataFrames: `source_customers`, `target_customers`.

In [ ]:
result = None  # Replace None with your DataFrame.

In [ ]:
check(result)